#  Big Data con PySpark — Notebook 5
## Spark SQL y Joins

---

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Vuelos_SQL")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

df = spark.read.parquet("/content/drive/MyDrive/Colab Notebooks/vuelos_limpio.parquet")
df.cache()
print(f"Filas: {df.count():,}")

Filas: 446,399


---
## 1. Spark SQL — Registrar DataFrame como tabla temporal

In [8]:
df.createOrReplaceTempView("vuelos")
# createOrReplaceTempView(nombre):
#   - Registra el DataFrame como una tabla virtual en el catálogo de Spark
#   - El nombre 'vuelos' es lo que usarás en las queries SQL
#   - 'TempView' = solo existe mientras la SparkSession esté activa
#   - 'OrReplace' = si ya existía una vista con ese nombre, la reemplaza
#   - No copia los datos, es solo un alias

# Verificar que la tabla fue registrada
spark.catalog.listTables()
# spark.catalog → gestor del catálogo de tablas y vistas
# .listTables() → devuelve la lista de tablas/vistas registradas

[Table(name='vuelos', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [9]:
# ── Ejecutar SQL con spark.sql() ──
resultado = spark.sql("""
    SELECT
        aerolinea,
        COUNT(*) AS total_vuelos,
        ROUND(AVG(tarifa_usd), 2) AS tarifa_promedio,
        ROUND(AVG(retraso_min), 1) AS retraso_promedio
    FROM vuelos
    WHERE estado != 'CANCELADO'
    GROUP BY aerolinea
    ORDER BY total_vuelos DESC
""")
# spark.sql(query_string) → ejecuta SQL estándar
# Devuelve un DataFrame de Spark (no un resultado inmediato)
# → lazy evaluation aplica: no ejecuta hasta .show(), .collect(), etc.

resultado.show()

+---------+------------+---------------+----------------+
|aerolinea|total_vuelos|tarifa_promedio|retraso_promedio|
+---------+------------+---------------+----------------+
|  Avianca|      144972|         425.14|            35.8|
|    LATAM|      104027|          424.3|            36.1|
|    Wingo|       62299|         424.37|            35.6|
|  EasyFly|       41412|         427.03|            35.9|
|   Satena|       41369|         423.79|            35.5|
|  JetBlue|       20953|         423.42|            35.3|
+---------+------------+---------------+----------------+



In [10]:
spark.sql("""
    SELECT
        origen,
        destino,
        ruta,
        total_vuelos,
        tarifa_promedio
    FROM (
        SELECT
            origen,
            destino,
            CONCAT(origen, '-', destino) AS ruta,
            COUNT(*) AS total_vuelos,
            ROUND(AVG(tarifa_usd), 2) AS tarifa_promedio
        FROM vuelos
        GROUP BY origen, destino
    ) rutas_agg
    WHERE total_vuelos > 1000
    ORDER BY total_vuelos DESC
    LIMIT 15
""").show(truncate=False)

+------+-------+-------+------------+---------------+
|origen|destino|ruta   |total_vuelos|tarifa_promedio|
+------+-------+-------+------------+---------------+
|LET   |PEI    |LET-PEI|5112        |422.85         |
|VVC   |CTG    |VVC-CTG|5107        |428.84         |
|MDE   |MTR    |MDE-MTR|5090        |424.71         |
|MTR   |LET    |MTR-LET|5083        |423.89         |
|MDE   |CTG    |MDE-CTG|5083        |425.52         |
|MDE   |VVC    |MDE-VVC|5082        |421.52         |
|CLO   |MDE    |CLO-MDE|5062        |427.46         |
|PEI   |LET    |PEI-LET|5061        |422.04         |
|SMR   |CTG    |SMR-CTG|5058        |426.59         |
|SMR   |MDE    |SMR-MDE|5056        |426.03         |
|MTR   |PEI    |MTR-PEI|5054        |421.23         |
|MDE   |BAQ    |MDE-BAQ|5041        |422.05         |
|SMR   |CLO    |SMR-CLO|5040        |424.89         |
|CLO   |MTR    |CLO-MTR|5037        |426.69         |
|PEI   |CLO    |PEI-CLO|5036        |423.99         |
+------+-------+-------+----

In [11]:
# ── SQL con CASE WHEN ──
spark.sql("""
    SELECT
        aerolinea,
        mes,
        COUNT(*) AS vuelos,
        ROUND(AVG(
            CASE
                WHEN estado = 'A_TIEMPO' THEN 1.0
                ELSE 0.0
            END
        ) * 100, 1) AS pct_puntualidad
    FROM vuelos
    GROUP BY aerolinea, mes
    HAVING COUNT(*) > 500
    ORDER BY aerolinea, mes
""").show(20)

+---------+---+------+---------------+
|aerolinea|mes|vuelos|pct_puntualidad|
+---------+---+------+---------------+
|  Avianca|  1| 13274|           67.7|
|  Avianca|  2| 11940|           68.1|
|  Avianca|  3| 13151|           67.7|
|  Avianca|  4| 13004|           68.3|
|  Avianca|  5| 13117|           68.8|
|  Avianca|  6| 12842|           68.6|
|  Avianca|  7| 13359|           67.7|
|  Avianca|  8| 13457|           67.6|
|  Avianca|  9| 12831|           68.2|
|  Avianca| 10| 13223|           67.4|
|  Avianca| 11| 12799|           68.5|
|  Avianca| 12| 13008|           68.1|
|  EasyFly|  1|  3822|           68.0|
|  EasyFly|  2|  3412|           68.5|
|  EasyFly|  3|  3908|           68.0|
|  EasyFly|  4|  3687|           67.4|
|  EasyFly|  5|  3824|           68.6|
|  EasyFly|  6|  3592|           68.2|
|  EasyFly|  7|  3746|           66.3|
|  EasyFly|  8|  3781|           66.4|
+---------+---+------+---------------+
only showing top 20 rows


In [15]:
# ── SQL con Window Functions usando CTE ──
spark.sql("""
    WITH rutas_ranked AS (
        SELECT
            aerolinea,
            CONCAT(origen, '-', destino) AS ruta,
            ROUND(AVG(tarifa_usd), 2) AS tarifa_prom_ruta,
            ROUND(AVG(AVG(tarifa_usd)) OVER (PARTITION BY aerolinea), 2) AS tarifa_prom_aerolinea,
            RANK() OVER (
                PARTITION BY aerolinea
                ORDER BY AVG(tarifa_usd) DESC
            ) AS rank_tarifa
        FROM vuelos
        GROUP BY aerolinea, origen, destino
    )
    SELECT
        aerolinea,
        ruta,
        tarifa_prom_ruta,
        tarifa_prom_aerolinea,
        rank_tarifa
    FROM rutas_ranked
    WHERE rank_tarifa <= 3
    ORDER BY aerolinea, rank_tarifa
""").show(30, truncate=False)

+---------+-------+----------------+---------------------+-----------+
|aerolinea|ruta   |tarifa_prom_ruta|tarifa_prom_aerolinea|rank_tarifa|
+---------+-------+----------------+---------------------+-----------+
|Avianca  |CLO-MDE|439.25          |424.95               |1          |
|Avianca  |LET-BAQ|439.01          |424.95               |2          |
|Avianca  |PEI-BAQ|438.08          |424.95               |3          |
|EasyFly  |CTG-MTR|452.62          |426.61               |1          |
|EasyFly  |LET-CLO|449.28          |426.61               |2          |
|EasyFly  |LET-SMR|448.52          |426.61               |3          |
|JetBlue  |VVC-LET|452.09          |423.44               |1          |
|JetBlue  |SMR-CLO|449.19          |423.44               |2          |
|JetBlue  |CLO-BOG|447.92          |423.44               |3          |
|LATAM    |BAQ-BOG|436.21          |424.55               |1          |
|LATAM    |MTR-VVC|435.24          |424.55               |2          |
|LATAM

---
## 2. Joins — Unir DataFrames

In [16]:
# ── Crear tabla de referencia de aeropuertos ──
aeropuertos_data = [
    ("BOG", "Bogotá",       "Cundinamarca",  2625),
    ("MDE", "Medellín",     "Antioquia",     1495),
    ("CLO", "Cali",         "Valle del Cauca", 995),
    ("CTG", "Cartagena",    "Bolívar",          2),
    ("BAQ", "Barranquilla", "Atlántico",        18),
    ("SMR", "Santa Marta",  "Magdalena",         4),
    ("PEI", "Pereira",      "Risaralda",      1342),
    ("VVC", "Villavicencio","Meta",            467),
    ("LET", "Leticia",      "Amazonas",         96),
    ("MTR", "Montería",     "Córdoba",          18),
]

df_aeropuertos = spark.createDataFrame(
    aeropuertos_data,
    schema=["codigo", "ciudad", "departamento", "altitud_msnm"]
)
# spark.createDataFrame(lista_de_tuplas, schema=[nombres_columnas])
# Crea un DataFrame Spark desde una lista Python
# schema puede ser lista de strings (nombres) o un StructType (con tipos)

df_aeropuertos.show()

+------+-------------+---------------+------------+
|codigo|       ciudad|   departamento|altitud_msnm|
+------+-------------+---------------+------------+
|   BOG|       Bogotá|   Cundinamarca|        2625|
|   MDE|     Medellín|      Antioquia|        1495|
|   CLO|         Cali|Valle del Cauca|         995|
|   CTG|    Cartagena|        Bolívar|           2|
|   BAQ| Barranquilla|      Atlántico|          18|
|   SMR|  Santa Marta|      Magdalena|           4|
|   PEI|      Pereira|      Risaralda|        1342|
|   VVC|Villavicencio|           Meta|         467|
|   LET|      Leticia|       Amazonas|          96|
|   MTR|     Montería|        Córdoba|          18|
+------+-------------+---------------+------------+



In [17]:
# ── INNER JOIN ──
df_con_ciudad = (
    df
    .join(
        df_aeropuertos,                    # DataFrame con el que unir
        df["origen"] == df_aeropuertos["codigo"],  # condición de unión
        how="inner"                        # tipo de join
        # inner → solo filas con match en ambos DataFrames
    )
    .select(
        df["vuelo_id"],
        df["origen"],
        df_aeropuertos["ciudad"].alias("ciudad_origen"),
        df_aeropuertos["departamento"].alias("depto_origen"),
        df["destino"],
        df["aerolinea"],
        df["tarifa_usd"]
    )
)
df_con_ciudad.show(5)

+--------+------+-------------+------------+-------+---------+----------+
|vuelo_id|origen|ciudad_origen|depto_origen|destino|aerolinea|tarifa_usd|
+--------+------+-------------+------------+-------+---------+----------+
|       4|   BAQ| Barranquilla|   Atlántico|    VVC|  EasyFly|    247.92|
|       6|   BAQ| Barranquilla|   Atlántico|    SMR|   Satena|    263.18|
|       7|   SMR|  Santa Marta|   Magdalena|    VVC|  EasyFly|    626.36|
|      12|   VVC|Villavicencio|        Meta|    BAQ|   Satena|    135.68|
|      13|   VVC|Villavicencio|        Meta|    LET|  Avianca|     360.2|
+--------+------+-------------+------------+-------+---------+----------+
only showing top 5 rows


In [18]:
# ── Tipos de Join ──
#
#  how='inner'      → solo filas con coincidencia en AMBOS DataFrames
#  how='left'       → todas las filas del LEFT, null donde no hay match
#  how='right'      → todas las filas del RIGHT, null donde no hay match
#  how='outer'      → todas las filas de AMBOS, null donde no hay match
#  how='left_semi'  → filas del LEFT que SÍ tienen match (sin columnas del RIGHT)
#  how='left_anti'  → filas del LEFT que NO tienen match
#  how='cross'      → producto cartesiano (todas × todas)

# ── LEFT ANTI JOIN: encontrar aeropuertos sin vuelos ──
aeropuertos_sin_vuelos = (
    df_aeropuertos
    .join(df, df_aeropuertos["codigo"] == df["origen"], how="left_anti")
)
print("Aeropuertos sin vuelos como origen:")
aeropuertos_sin_vuelos.show()

Aeropuertos sin vuelos como origen:
+------+------+------------+------------+
|codigo|ciudad|departamento|altitud_msnm|
+------+------+------------+------------+
+------+------+------------+------------+



In [19]:
# ── Broadcast Join — Optimización para tablas pequeñas ──
# Cuando uno de los DataFrames es pequeño (<= 10 MB típicamente),
# podemos enviarlo a todos los workers y evitar el shuffle

df_con_ciudad_broadcast = (
    df
    .join(
        F.broadcast(df_aeropuertos),  # F.broadcast() → marca para enviar a todos los nodos
        df["origen"] == df_aeropuertos["codigo"],
        how="inner"
    )
    .select(df["*"], df_aeropuertos["ciudad"].alias("ciudad_origen"))
)
print("Con broadcast join:")
df_con_ciudad_broadcast.show(5)

# ¿Por qué usar broadcast?
# Sin broadcast: Spark mueve datos de AMBAS tablas por la red (shuffle)
# Con broadcast: la tabla pequeña se envía UNA VEZ a cada worker
#                → evita el shuffle de la tabla grande → mucho más rápido

Con broadcast join:
+--------+---------+------+-------+---------+------------+-----------+--------+----------+---------+----+---+----+-------------+
|vuelo_id|aerolinea|origen|destino|pasajeros|distancia_km|retraso_min|  estado|tarifa_usd|    clase|anio|mes|hora|ciudad_origen|
+--------+---------+------+-------+---------+------------+-----------+--------+----------+---------+----+---+----+-------------+
|       1|    LATAM|   LET|    PEI|      144|        1977|          0|A_TIEMPO|     72.09|Economica|2023| 10|   3|      Leticia|
|       2|    LATAM|   CTG|    PEI|      135|         611|          0|A_TIEMPO|    478.33|Economica|2022|  2|  20|    Cartagena|
|       3|    Wingo|   MDE|    VVC|      117|        1082|          0|DEMORADO|    498.33|  Primera|2022|  8|  14|     Medellín|
|       4|  EasyFly|   BAQ|    VVC|      128|        1952|        235|A_TIEMPO|    247.92|Economica|2023|  5|  12| Barranquilla|
|       5|  Avianca|   MTR|    BOG|      150|        1756|          2|A_TIEMP

---
## 3. DataFrame API vs Spark SQL — ¿Cuál usar?

| Situación | Recomendación |
|---|---|
| Analistas que ya saben SQL | Spark SQL |
| Pipelines en producción | DataFrame API (más testeable) |
| Queries ad-hoc exploratorias | Spark SQL |
| Lógica compleja encadenada | DataFrame API |
| Ambas | **Son equivalentes en rendimiento** — Catalyst optimiza igual |

**Dato clave:** Internamente, ambas APIs generan el mismo plan de ejecución optimizado. La elección es de estilo, no de performance.

In [20]:
# ── Ver el plan de ejecución ──
query = spark.sql("""
    SELECT aerolinea, COUNT(*) AS vuelos
    FROM vuelos
    WHERE estado = 'DEMORADO'
    GROUP BY aerolinea
""")

query.explain(mode="simple")
# .explain() → muestra el plan físico de ejecución
# mode='simple'   → plan físico simplificado
# mode='extended' → plan lógico + plan físico
# mode='cost'     → incluye estimaciones de costo
# Útil para entender qué está haciendo Spark y detectar cuellos de botella

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[aerolinea#823], functions=[count(1)])
   +- Exchange hashpartitioning(aerolinea#823, 8), ENSURE_REQUIREMENTS, [plan_id=1100]
      +- HashAggregate(keys=[aerolinea#823], functions=[partial_count(1)])
         +- Project [aerolinea#823]
            +- Filter (isnotnull(estado#829) AND (estado#829 = DEMORADO))
               +- InMemoryTableScan [aerolinea#823, estado#829], [isnotnull(estado#829), (estado#829 = DEMORADO)]
                     +- InMemoryRelation [vuelo_id#822, aerolinea#823, origen#824, destino#825, pasajeros#826, distancia_km#827, retraso_min#828, estado#829, tarifa_usd#830, clase#831, anio#832, mes#833, hora#834], StorageLevel(disk, memory, deserialized, 1 replicas)
                           +- *(1) ColumnarToRow
                              +- FileScan parquet [vuelo_id#0,aerolinea#1,origen#2,destino#3,pasajeros#4,distancia_km#5,retraso_min#6,estado#7,tarifa_usd#8,clase#9,anio#10,mes#11,h

---
## Resumen del notebook

```
df.createOrReplaceTempView("nombre")     →  registrar como tabla SQL
spark.sql("SELECT ... FROM nombre ...")  →  ejecutar SQL estándar
spark.createDataFrame(lista, schema)     →  crear DF desde Python

df.join(df2, condicion, how='inner')     →  inner join
df.join(df2, condicion, how='left')      →  left join
df.join(df2, condicion, how='left_anti') →  filas sin match
F.broadcast(df_pequeño)                  →  optimizar joins con tablas chicas

query.explain(mode='simple')             →  ver plan de ejecución
```

**Siguiente:** Pipeline completo de análisis